# Notebook 02 — Stage O: LP Dispatch Optimization

This notebook implements Stage O of the M-S-O pipeline:
1. Read `lambda_hat.csv` from Stage M
2. Load distance matrix and LP configuration
3. Solve one integer LP per hourly window (t=0..23) using PuLP/CBC
4. Save the assignment tensor `lp_assignment.json` for Stage S

**Prerequisite:** Run `notebook_01_forecasting.ipynb` first.

In [ ]:
import sys
from pathlib import Path

repo_root = Path().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.utils.config_loader import load_config
from src.utils.distance import load_distance_matrix, flight_time_matrix
from src.optimization.lp_formulation import LPDispatchSolver

print('Imports successful')

## 1. Load Inputs

In [ ]:
# Load configs
network_cfg = load_config(repo_root / 'config' / 'network.yaml')
lp_cfg = load_config(repo_root / 'config' / 'lp.yaml')
sim_cfg = load_config(repo_root / 'config' / 'simulation.yaml')

# Load lambda_hat from Stage M
lambda_hat_path = repo_root / 'data' / 'processed' / 'lambda_hat.csv'
lh_df = pd.read_csv(lambda_hat_path, index_col=0)
lambda_hat = lh_df.values  # shape (12, 24)
print(f'lambda_hat shape: {lambda_hat.shape}')

# Load distance matrix
d_km = load_distance_matrix(network_cfg)  # shape (3, 12)
speed_kmh = float(sim_cfg['drone']['speed_kmh'])
d_min = flight_time_matrix(d_km, speed_kmh)  # convert to minutes
print(f'Distance matrix (km):\n{d_km}')
print(f'Flight time matrix (min):\n{d_min.round(2)}')

# Initial inventory: 50 units per type x 4 types per bank
init_per_type = int(sim_cfg['inventory']['initial_per_bank_per_type'])
n_types = len(sim_cfg['inventory']['blood_types'])
I_init = np.full(3, float(init_per_type * n_types))
print(f'\nInitial inventory per bank: {I_init}')
print(f'C_fleet: {lp_cfg["fleet"]["C_fleet"]}')

## 2. Solve LP for All 24 Windows

In [ ]:
print('Solving LP for windows t=0..23 (should be < 1 second)...')

solver = LPDispatchSolver(lp_cfg)
x_sol = solver.solve(lambda_hat, d_min, I_init)

# Count optimal vs infeasible windows
T = 24
B, H = 3, 12
optimal_windows = set()
infeasible_windows = set()
for t in range(T):
    val = x_sol.get((0, 0, t), 0)
    if val == -1:
        infeasible_windows.add(t)
    else:
        optimal_windows.add(t)

print(f'Optimal windows:    {len(optimal_windows)}/24')
print(f'Infeasible windows: {len(infeasible_windows)}/24')

## 3. Inspect Assignment Matrix

In [ ]:
# Show assignment totals per window
window_totals = []
for t in range(T):
    total = sum(max(0, x_sol.get((b, h, t), 0)) for b in range(B) for h in range(H))
    window_totals.append(total)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Total assignments per window
ax1.bar(range(T), window_totals, color='steelblue', edgecolor='black')
ax1.axhline(lp_cfg['fleet']['C_fleet'], color='red', linestyle='--', label=f'C_fleet={lp_cfg["fleet"]["C_fleet"]}')
ax1.set_xlabel('Hour (window t)')
ax1.set_ylabel('Total Assignments')
ax1.set_title('LP Total Assignments per Window')
ax1.legend()

# Heatmap for window t=14
t_show = 14
mat = np.array([[max(0, x_sol.get((b, h, t_show), 0)) for h in range(H)] for b in range(B)])
sns.heatmap(mat, ax=ax2, annot=True, fmt='.0f', cmap='Blues',
            xticklabels=[f'H{h+1:02d}' for h in range(H)],
            yticklabels=[f'Bank {b+1}' for b in range(B)])
ax2.set_title(f'LP Assignment Matrix x[b,h] — Window t={t_show}')
ax2.set_xlabel('Hospital')
ax2.set_ylabel('Blood Bank')

plt.tight_layout()
plt.show()

## 4. Verify Constraints

In [ ]:
import math

C_fleet = lp_cfg['fleet']['C_fleet']
violations = []

for t in optimal_windows:
    # Fleet cap
    total = sum(max(0, x_sol.get((b, h, t), 0)) for b in range(B) for h in range(H))
    if total > C_fleet:
        violations.append(f't={t}: fleet cap violated ({total} > {C_fleet})')
    # Demand coverage
    for h in range(H):
        assigned = sum(max(0, x_sol.get((b, h, t), 0)) for b in range(B))
        demand = math.ceil(lambda_hat[h, t])
        if assigned < demand:
            violations.append(f't={t}, h={h}: demand not met ({assigned} < {demand})')

if violations:
    print(f'CONSTRAINT VIOLATIONS FOUND ({len(violations)}):')
    for v in violations[:10]:
        print(f'  {v}')
else:
    print('All LP constraints satisfied for optimal windows.')

## 5. Save LP Assignment

In [ ]:
out_path = repo_root / 'data' / 'processed' / 'lp_assignment.json'
solver.save(out_path)
print(f'LP assignment saved to {out_path}')

# Verify can be reloaded
x_sol_loaded = LPDispatchSolver.load(out_path)
print(f'Reloaded {len(x_sol_loaded)} entries from JSON')
assert len(x_sol_loaded) == B * H * T, 'Entry count mismatch'
print('Load verification passed.')

## Stage O Complete

**Outputs saved:**
- `data/processed/lp_assignment.json` ← input to Stage S (LP-Optimized policy)

**Next:** Run `notebook_03_simulation.ipynb`